# Fair training and evaluation on CIFAR-10 / CIFAR-100

Notebook này huấn luyện sáu kiến trúc với **một recipe augmentation chung trong từng thí nghiệm**. Giá trị trong YAML là mặc định; `EPOCHS_OVERRIDE` trong bảng điều khiển có quyền ưu tiên khi được đặt.

Luồng train và test-eval được tách bằng switch. Train chỉ dùng train/validation để chọn `best.pth`; test chỉ chạy khi `RUN_EVAL = True`.

In [ ]:
from __future__ import annotations

import copy
import json
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd
import torch
import yaml
from IPython.display import display

# Đặt đường dẫn tuyệt đối nếu notebook không nằm trong repository.
REPO_ROOT_OVERRIDE = None

def locate_repo(override=None):
    candidates = []
    if override:
        candidates.append(Path(override).expanduser())
    env_root = os.environ.get('HBCC_REPO_ROOT')
    if env_root:
        candidates.append(Path(env_root).expanduser())
    cwd = Path.cwd()
    candidates.extend([
        cwd,
        cwd / 'Lightweight-Context-Cluster',
        Path('/kaggle/working/Lightweight-Context-Cluster'),
    ])
    candidates.extend(cwd.parents)
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'lightweight_hbcc').is_dir() and (candidate / 'tools/train.py').is_file():
            return candidate
    raise FileNotFoundError(
        'Không tìm thấy repository. Hãy đặt REPO_ROOT_OVERRIDE tới thư mục Lightweight-Context-Cluster.'
    )

ROOT = locate_repo(REPO_ROOT_OVERRIDE)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lightweight_hbcc.config import deep_update, load_config, save_config
from lightweight_hbcc.data import build_loaders
from lightweight_hbcc.engine import evaluate, load_checkpoint, resolve_device
from lightweight_hbcc.models import build_model

print('Repository:', ROOT)
print('Python    :', sys.executable)
print('PyTorch   :', torch.__version__)
print('CUDA      :', torch.cuda.is_available())

## 1. Bảng điều khiển

- `EPOCHS_OVERRIDE = None`: dùng số epoch mặc định trong YAML (300 với recipe HBCC cũ).
- Đặt `EPOCHS_OVERRIDE = 100`, `200`, `300`, ... để ghi đè đồng thời training và metadata protocol.
- `None` tự chọn `hbcc_legacy` cho cả CIFAR-10 và CIFAR-100, đúng recipe trong bảng báo cáo.
- Mọi profile chỉ hợp lệ khi áp dụng lại cho **tất cả** model trong cùng bảng so sánh.
- Các switch chạy tốn tài nguyên mặc định đều tắt để tránh vô tình chạy sáu model khi bấm Run All.

In [ ]:
# ===== DATASET / RECIPE =====
DATASET = 'cifar100'                 # 'cifar10' hoặc 'cifar100'
AUGMENTATION_PROFILE = None         # None = default theo dataset; hoặc chọn profile cụ thể bên dưới
EPOCHS_OVERRIDE = None              # None = mặc định YAML; hoặc số nguyên dương
SEED = 17

# ===== MODEL SWITCHES =====
MODEL_SWITCHES = {
    'resnet18': True,
    'mobilenet_v2': True,
    'shufflenet_v2_x1_0': True,
    'coc_baseline': True,
    'hbcc_small': True,
    'hbcc_medium': True,
    # Spatial ablations: enable one candidate at a time against its HBCC baseline.
    'hbcc_small_keep4': False,
    'hbcc_medium_keep4': False,
    'hbcc_medium_keep4_late_hybrid': False,
}

# ===== PROCESS SWITCHES =====
RUN_DATA_PREP = True                # tải/kiểm tra dữ liệu đúng một lần
RUN_PREFLIGHT = True                # kiểm tra fairness + forward shape + params
RUN_TRAIN = False                   # train/validation, không chạm test set
RUN_EVAL = False                    # đánh giá best.pth trên official test split
RUN_SUMMARY = True                  # tổng hợp các kết quả đang có

# ===== RUNTIME =====
SMOKE_TEST = False                  # 1 epoch, 1 batch train/val/test
DOWNLOAD_IF_MISSING = True          # Kaggle cần bật Internet nếu dữ liệu chưa có
DATA_ROOT_OVERRIDE = None           # phải là thư mục CHA của cifar-*-...
OUTPUT_ROOT_OVERRIDE = None
WORKERS_OVERRIDE = 2
DEVICE = 'auto'
SHOW_PROGRESS = False
PRINT_EVERY = 5
SKIP_COMPLETED = True
FORCE_RETRAIN = False

assert DATASET in {'cifar10', 'cifar100'}
DEFAULT_PROFILES = {'cifar10': 'hbcc_legacy', 'cifar100': 'hbcc_legacy'}
AUGMENTATION_PROFILE = AUGMENTATION_PROFILE or DEFAULT_PROFILES[DATASET]
if EPOCHS_OVERRIDE is not None:
    assert isinstance(EPOCHS_OVERRIDE, int) and EPOCHS_OVERRIDE > 0
assert isinstance(SEED, int)
SELECTED_MODELS = [name for name, enabled in MODEL_SWITCHES.items() if enabled]
assert SELECTED_MODELS, 'Hãy bật ít nhất một model.'
print('Selected models:', SELECTED_MODELS)

## 2. Chuẩn bị dữ liệu

Notebook ưu tiên dữ liệu đã có trong `DATA_ROOT_OVERRIDE`, repository hoặc Kaggle Input. Nếu không tìm thấy, dữ liệu được tải một lần vào thư mục writable; mọi run sau đó dùng `download: false`.

In [ ]:
from torchvision import datasets

DATASET_CLASS = datasets.CIFAR10 if DATASET == 'cifar10' else datasets.CIFAR100
DATA_MARKER = 'cifar-10-batches-py' if DATASET == 'cifar10' else 'cifar-100-python'

def marker_exists(root):
    return (Path(root) / DATA_MARKER).is_dir()

def find_kaggle_root():
    kaggle_input = Path('/kaggle/input')
    if not kaggle_input.is_dir():
        return None
    patterns = [f'*/{DATA_MARKER}', f'*/*/{DATA_MARKER}', f'*/*/*/{DATA_MARKER}']
    for pattern in patterns:
        for marker in kaggle_input.glob(pattern):
            if marker.is_dir():
                return marker.parent
    return None

if DATA_ROOT_OVERRIDE is not None:
    DATA_ROOT = Path(DATA_ROOT_OVERRIDE).expanduser().resolve()
elif marker_exists(ROOT / 'data'):
    DATA_ROOT = (ROOT / 'data').resolve()
else:
    DATA_ROOT = find_kaggle_root() or (ROOT / 'data').resolve()

if RUN_DATA_PREP:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    # download=True vẫn kiểm tra integrity; nếu dữ liệu hợp lệ torchvision không tải lại.
    # Nếu thư mục marker tồn tại nhưng thiếu/hỏng file, torchvision có thể tự sửa khi Internet khả dụng.
    should_download = DOWNLOAD_IF_MISSING
    try:
        DATASET_CLASS(root=str(DATA_ROOT), train=True, download=should_download)
        DATASET_CLASS(root=str(DATA_ROOT), train=False, download=should_download)
    except Exception as exc:
        raise RuntimeError(
            f'Không thể chuẩn bị {DATASET} tại {DATA_ROOT}. Nếu Kaggle Internet tắt, '
            f'hãy attach dataset đã giải nén và đặt DATA_ROOT_OVERRIDE tới thư mục cha của {DATA_MARKER}.'
        ) from exc
else:
    try:
        DATASET_CLASS(root=str(DATA_ROOT), train=True, download=False)
        DATASET_CLASS(root=str(DATA_ROOT), train=False, download=False)
    except Exception as exc:
        raise RuntimeError(
            f'{DATASET} tại {DATA_ROOT} thiếu hoặc hỏng. Bật RUN_DATA_PREP để sửa/chuẩn bị lại.'
        ) from exc

if not marker_exists(DATA_ROOT):
    raise RuntimeError(
        f'Thiếu {DATA_ROOT / DATA_MARKER}. Bật RUN_DATA_PREP hoặc đặt DATA_ROOT_OVERRIDE chính xác.'
    )

print('Dataset  :', DATASET)
print('Data root:', DATA_ROOT)
print('Marker   :', DATA_ROOT / DATA_MARKER)

## 3. Tạo effective config

Mọi model nhận bản sao của cùng `protocol`, `data` và `train`. Chỉ `model` và `experiment.name` được phép khác nhau. Effective config được lưu cạnh output để có thể tái lập chính xác.

In [ ]:
PROFILE_FILES = {
    'cifar10': {
        'hbcc_legacy': ROOT / 'configs/cifar_fair/cifar10_hbcc_legacy.yaml',
        'primary': ROOT / 'configs/cifar_fair/cifar10_primary.yaml',
        'mild_erasing': ROOT / 'configs/cifar_fair/cifar10_mild_erasing.yaml',
    },
    'cifar100': {
        'hbcc_legacy': ROOT / 'configs/cifar_fair/cifar100_hbcc_legacy.yaml',
        'primary': ROOT / 'configs/cifar_fair/cifar100_primary.yaml',
        'mild_cutmix': ROOT / 'configs/cifar_fair/cifar100_mild_cutmix.yaml',
    },
}
if AUGMENTATION_PROFILE not in PROFILE_FILES[DATASET]:
    raise ValueError(f'Profile hợp lệ cho {DATASET}: {list(PROFILE_FILES[DATASET])}')

RECIPE_PATH = PROFILE_FILES[DATASET][AUGMENTATION_PROFILE]
CATALOG_PATH = ROOT / 'configs/cifar_fair/model_catalog.yaml'
recipe = load_config(RECIPE_PATH)
catalog = yaml.safe_load(CATALOG_PATH.read_text(encoding='utf-8'))['models']
unknown = sorted(set(SELECTED_MODELS) - set(catalog))
if unknown:
    raise KeyError(f'Model không có trong catalog: {unknown}')

default_epochs = int(recipe['train']['epochs'])
assert default_epochs == int(recipe['protocol']['effective_epochs'])
effective_epochs = 1 if SMOKE_TEST else (default_epochs if EPOCHS_OVERRIDE is None else EPOCHS_OVERRIDE)
run_variant = 'smoke' if SMOKE_TEST else f'e{effective_epochs}'
num_classes = 10 if DATASET == 'cifar10' else 100
is_canonical = bool(recipe['protocol'].get('canonical', False)) and not SMOKE_TEST and effective_epochs == default_epochs
protocol_name = recipe['protocol']['name']
if SMOKE_TEST or effective_epochs != default_epochs:
    suffix = 'smoke' if SMOKE_TEST else f'e{effective_epochs}'
    protocol_name = f'{protocol_name}_{suffix}'

effective_recipe = deep_update(recipe, {
    'protocol': {
        'name': protocol_name,
        'canonical': is_canonical,
        'effective_epochs': effective_epochs,
    },
    'data': {
        'root': str(DATA_ROOT),
        'download': False,
        'loader_seed': SEED,
        'workers': WORKERS_OVERRIDE,
    },
    'train': {
        'seed': SEED,
        'epochs': effective_epochs,
    },
})

OUTPUT_ROOT = (
    Path(OUTPUT_ROOT_OVERRIDE).expanduser().resolve()
    if OUTPUT_ROOT_OVERRIDE is not None
    else (ROOT / 'runs_cifar_fair').resolve()
)
CONFIG_OUTPUT = OUTPUT_ROOT / '_effective_configs' / DATASET / AUGMENTATION_PROFILE / run_variant
CONFIG_OUTPUT.mkdir(parents=True, exist_ok=True)

RUNS = {}
profile_model_overrides = recipe.get('model_overrides', {})
for model_key in SELECTED_MODELS:
    model_cfg = copy.deepcopy(catalog[model_key]['model'])
    model_cfg = deep_update(model_cfg, profile_model_overrides.get(model_key, {}))
    model_cfg['num_classes'] = num_classes
    run_name = f'fair_{DATASET}_{AUGMENTATION_PROFILE}_{model_key}_seed{SEED}_{run_variant}'
    cfg = deep_update(effective_recipe, {
        'experiment': {'name': run_name},
        'model': model_cfg,
    })
    config_path = CONFIG_OUTPUT / f'{run_name}.yaml'
    save_config(cfg, config_path)
    RUNS[model_key] = {
        'display_name': catalog[model_key]['display_name'],
        'config': cfg,
        'config_path': config_path,
        'run_name': run_name,
        'run_dir': OUTPUT_ROOT / run_name,
    }

print('Recipe YAML     :', RECIPE_PATH)
print('Profile         :', AUGMENTATION_PROFILE)
print('Default epochs  :', default_epochs)
print('Effective epochs:', effective_epochs)
print('Protocol        :', protocol_name)
print('Output root     :', OUTPUT_ROOT)

## 4. Fairness preflight

Cell này fail-fast nếu một model nhận augmentation, optimizer, epoch, split hoặc seed khác các model còn lại.

In [ ]:
PARAM_COUNTS = {}
if RUN_PREFLIGHT:
    reference_shared = None
    rows = []
    for model_key, spec in RUNS.items():
        cfg = load_config(spec['config_path'])
        shared = {key: cfg[key] for key in ('protocol', 'data', 'train')}
        if reference_shared is None:
            reference_shared = shared
        elif shared != reference_shared:
            raise AssertionError(f'{model_key} không dùng cùng shared recipe.')
        model = build_model(cfg).eval()
        params = sum(parameter.numel() for parameter in model.parameters())
        with torch.inference_mode():
            output = model(torch.randn(1, 3, 32, 32))
        if tuple(output.shape) != (1, num_classes):
            raise AssertionError(f'{model_key}: output shape={tuple(output.shape)}')
        PARAM_COUNTS[model_key] = params
        rows.append({
            'model': model_key,
            'display_name': spec['display_name'],
            'params': params,
            'params_M': round(params / 1e6, 3),
            'output': str(tuple(output.shape)),
        })
        del model
    preflight_df = pd.DataFrame(rows)
    display(preflight_df)
    print('Fairness preflight: PASS - protocol/data/train giống hệt giữa mọi model.')
else:
    print('RUN_PREFLIGHT=False: đã bỏ qua preflight.')

## 5. Training switch

Training luôn truyền `--skip-test`; vì vậy `RUN_TRAIN=True, RUN_EVAL=False` không sử dụng test set. Notebook không resume controlled runs vì checkpoint hiện chưa lưu đầy đủ scheduler/scaler/RNG.

In [ ]:
def read_epoch_records(metrics_path):
    if not metrics_path.is_file():
        return []
    records = []
    for line in metrics_path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            record = json.loads(line)
            if isinstance(record.get('epoch'), int) and 'val_acc1' in record:
                records.append(record)
    return records

def training_complete(run_dir, epochs):
    records = read_epoch_records(run_dir / 'metrics.jsonl')
    return (
        (run_dir / 'config.yaml').is_file()
        and (run_dir / 'best.pth').is_file()
        and (run_dir / 'latest.pth').is_file()
        and records
        and max(record['epoch'] for record in records) == epochs - 1
    )

def stored_config_compatible(run_dir, expected_cfg):
    stored_path = run_dir / 'config.yaml'
    if not stored_path.is_file():
        return False
    stored = load_config(stored_path)
    expected = copy.deepcopy(expected_cfg)
    # Vị trí vật lý của cùng dataset có thể đổi giữa local và Kaggle.
    stored['data']['root'] = '<DATA_ROOT>'
    expected['data']['root'] = '<DATA_ROOT>'
    return stored == expected

if RUN_TRAIN:
    for model_key, spec in RUNS.items():
        run_dir = spec['run_dir']
        complete = training_complete(run_dir, effective_epochs)
        compatible = stored_config_compatible(run_dir, spec['config'])
        if complete and compatible and SKIP_COMPLETED and not FORCE_RETRAIN:
            print(f'[skip] completed: {spec["run_name"]}')
            continue
        if run_dir.exists() and not FORCE_RETRAIN:
            raise FileExistsError(
                f'{run_dir} đã tồn tại nhưng incomplete hoặc config không tương thích. '
                'Đặt FORCE_RETRAIN=True hoặc đổi output/epoch.'
            )
        command = [
            sys.executable,
            str(ROOT / 'tools/train.py'),
            '--config', str(spec['config_path']),
            '--output', str(OUTPUT_ROOT),
            '--device', DEVICE,
            '--print-every', str(PRINT_EVERY),
            '--skip-test',
        ]
        if not SHOW_PROGRESS:
            command.append('--no-progress')
        if SMOKE_TEST:
            command.extend([
                '--limit-train-batches', '1',
                '--limit-val-batches', '1',
            ])
        print('\n[train]', spec['run_name'])
        subprocess.run(command, cwd=ROOT, check=True)
else:
    print('RUN_TRAIN=False: không chạy training.')

## 6. Evaluation switch

Evaluation nạp `config.yaml` đã được lưu trong từng run và `best.pth`, xác nhận dataset/profile/epoch/seed trước khi đánh giá official test split. Một test loader chung được tái sử dụng cho mọi model.

In [ ]:
if RUN_EVAL:
    eval_data_cfg = copy.deepcopy(effective_recipe['data'])
    eval_data_cfg['download'] = False
    _, _, test_loader = build_loaders(eval_data_cfg, include_test=True)
    if test_loader is None:
        raise RuntimeError('Không tạo được test loader.')
    device = resolve_device(DEVICE)
    print('Evaluation device:', device)
    for model_key, spec in RUNS.items():
        run_dir = spec['run_dir']
        stored_config_path = run_dir / 'config.yaml'
        checkpoint_path = run_dir / 'best.pth'
        if not stored_config_path.is_file() or not checkpoint_path.is_file():
            raise FileNotFoundError(f'Thiếu config/checkpoint cho {model_key}: {run_dir}')
        stored_cfg = load_config(stored_config_path)
        expected_cfg = spec['config']
        for key in ('protocol', 'train', 'model', 'experiment'):
            if stored_cfg.get(key) != expected_cfg.get(key):
                raise AssertionError(f'{model_key}: stored {key} không khớp lựa chọn notebook.')
        stored_data = copy.deepcopy(stored_cfg['data'])
        expected_data = copy.deepcopy(expected_cfg['data'])
        stored_data['root'] = expected_data['root']
        stored_data['download'] = False
        if stored_data != expected_data:
            raise AssertionError(f'{model_key}: stored data recipe không khớp.')
        model = build_model(stored_cfg).to(device)
        checkpoint = load_checkpoint(model, checkpoint_path, device, strict=True)
        metrics = evaluate(
            model,
            test_loader,
            device,
            amp=bool(stored_cfg['train'].get('amp', True)),
            limit_batches=1 if SMOKE_TEST else None,
            progress=SHOW_PROGRESS,
            prefix='test',
        )
        record = {
            'phase': 'test',
            'model': model_key,
            'dataset': DATASET,
            'augmentation_profile': AUGMENTATION_PROFILE,
            'seed': SEED,
            'effective_epochs': effective_epochs,
            'protocol_name': protocol_name,
            'epoch': int(checkpoint.get('best_epoch', -1)),
            'checkpoint': 'best.pth',
            **metrics,
        }
        (run_dir / 'test_metrics.json').write_text(
            json.dumps(record, indent=2, ensure_ascii=False) + '\n',
            encoding='utf-8',
        )
        print(model_key, json.dumps(metrics))
        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()
else:
    print('RUN_EVAL=False: không chạy test evaluation.')

## 7. Tổng hợp kết quả

Bảng vẫn hiển thị best validation khi test chưa được bật. Kết quả dùng epoch override không bị loại bỏ; tên output chứa số epoch để tránh va chạm.

In [ ]:
if RUN_SUMMARY:
    rows = []
    for model_key, spec in RUNS.items():
        run_dir = spec['run_dir']
        epoch_records = read_epoch_records(run_dir / 'metrics.jsonl')
        best_record = max(epoch_records, key=lambda row: row['val_acc1']) if epoch_records else {}
        test_path = run_dir / 'test_metrics.json'
        test_record = json.loads(test_path.read_text(encoding='utf-8')) if test_path.is_file() else {}
        params = PARAM_COUNTS.get(model_key)
        if params is None:
            params = sum(p.numel() for p in build_model(spec['config']).parameters())
        status = 'evaluated' if test_record else ('trained' if epoch_records else 'not_run')
        rows.append({
            'model': model_key,
            'display_name': spec['display_name'],
            'params_M': round(params / 1e6, 3),
            'best_val_acc1': best_record.get('val_acc1'),
            'best_val_epoch': best_record.get('epoch', -1) + 1 if best_record else None,
            'test_acc1': test_record.get('test_acc1'),
            'test_acc5': test_record.get('test_acc5'),
            'dataset': DATASET,
            'profile': AUGMENTATION_PROFILE,
            'epochs': effective_epochs,
            'seed': SEED,
            'canonical': is_canonical,
            'status': status,
            'run_dir': str(run_dir),
        })
    summary_df = pd.DataFrame(rows).sort_values(
        ['test_acc1', 'best_val_acc1', 'params_M'],
        ascending=[False, False, True],
        na_position='last',
    )
    summary_path = OUTPUT_ROOT / f'summary_{DATASET}_{AUGMENTATION_PROFILE}_seed{SEED}_{run_variant}.csv'
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(summary_path, index=False)
    display(summary_df)
    print('Summary:', summary_path)
else:
    print('RUN_SUMMARY=False: không tổng hợp kết quả.')

## Quy tắc báo cáo

1. Chọn dataset, augmentation profile và epoch **trước** khi chạy ma trận model.
2. Không đổi augmentation riêng cho HBCC hoặc một baseline.
3. `best.pth` được chọn bằng validation split; official test chỉ chạy trong cell evaluation.
4. Với recipe `hbcc_legacy`, mọi batch dùng MixUp hoặc CutMix; `train_acc1` so với nhãn gốc chỉ là diagnostic. Đánh giá bằng validation curve và `best.pth`.
5. Với một seed, kết quả chỉ mang tính mô tả; không báo cáo độ lệch chuẩn hoặc khoảng tin cậy.
6. Nếu đổi epoch/profile, dùng bảng kết quả riêng vì protocol không còn giống bảng trước.